In [1]:
import pandas as pd

In [2]:
pd.set_option("display.max_rows", 1000000)

In [3]:
 # pd.set_option("display.max_colwidth", None)

# Customer

In [4]:
customer = pd.read_csv("../data/olist_customers_dataset.csv")

In [5]:
customer.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [6]:
# Check Null
customer.isnull().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [7]:
customer.count()

customer_id                 99441
customer_unique_id          99441
customer_zip_code_prefix    99441
customer_city               99441
customer_state              99441
dtype: int64

In [8]:
# Check duplicate
customer.drop_duplicates().count()

customer_id                 99441
customer_unique_id          99441
customer_zip_code_prefix    99441
customer_city               99441
customer_state              99441
dtype: int64

In [9]:
# Loại bỏ khoảng trắng, đổi thành chữ hoa đầu dòng cột customer_city
customer["customer_city"] = customer["customer_city"].str.strip().str.title()

In [10]:
# Thêm số 0 vào đầu cho các code có độ dài < 5
customer["customer_zip_code_prefix"] = customer["customer_zip_code_prefix"].astype(str) \
                                                                           .str.zfill(5) 
                                                                         

In [11]:
customer.duplicated().sum()

np.int64(0)

In [12]:
customer.head(5)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,Franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,Sao Bernardo Do Campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,Sao Paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,Mogi Das Cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,Campinas,SP


In [13]:
customer.to_csv("../clean/customers.csv", index = False, encoding = "utf-8")

# Geolocation

In [14]:
geolocation = pd.read_csv("../data/olist_geolocation_dataset.csv")

In [15]:
geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [16]:
# Check Null
geolocation.isnull().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [17]:
geolocation.count()

geolocation_zip_code_prefix    1000163
geolocation_lat                1000163
geolocation_lng                1000163
geolocation_city               1000163
geolocation_state              1000163
dtype: int64

In [18]:
# Check Duplicate
geolocation.drop_duplicates().count()

geolocation_zip_code_prefix    738332
geolocation_lat                738332
geolocation_lng                738332
geolocation_city               738332
geolocation_state              738332
dtype: int64

In [19]:
geo_dup = geolocation.drop_duplicates()

In [20]:
import unidecode

In [21]:
# Loại bỏ khoảng trắng, đổi thành chữ hoa đầu dòng cột geolocation_city
geo_dup["geolocation_city"] = geo_dup["geolocation_city"].str.strip().str.title()
geo_dup["geolocation_city"] = geo_dup["geolocation_city"].apply(unidecode.unidecode)

In [22]:
geo_dup["geolocation_zip_code_prefix"].duplicated().sum()

np.int64(719317)

In [23]:
geo_dup.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,Sao Paulo,SP
1,1046,-23.546081,-46.644820,Sao Paulo,SP
2,1046,-23.546129,-46.642951,Sao Paulo,SP
3,1041,-23.544392,-46.639499,Sao Paulo,SP
4,1035,-23.541578,-46.641607,Sao Paulo,SP


In [24]:
# Nhóm các bản ghi code bị dup
geo_clean = geo_dup \
                .groupby("geolocation_zip_code_prefix") \
                .agg({
                    "geolocation_lat" : "mean",
                    "geolocation_lng" : "mean", 
                    "geolocation_city" : "first",
                    "geolocation_state" : "first" 
                }) \
                .reset_index()

In [25]:
geo_clean.head(5)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1001,-23.550227,-46.634039,Sao Paulo,SP
1,1002,-23.547657,-46.634991,Sao Paulo,SP
2,1003,-23.549000,-46.635582,Sao Paulo,SP
3,1004,-23.549829,-46.634792,Sao Paulo,SP
4,1005,-23.549547,-46.636406,Sao Paulo,SP


In [26]:
# Thêm số 0 vào đầu cho các code có độ dài < 5
geo_clean["geolocation_zip_code_prefix"] = geo_clean["geolocation_zip_code_prefix"].astype(str) \
                                                                           .str.zfill(5) 
                                                                         

In [27]:
geo_clean.head(5)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01001,-23.550227,-46.634039,Sao Paulo,SP
1,01002,-23.547657,-46.634991,Sao Paulo,SP
2,01003,-23.549000,-46.635582,Sao Paulo,SP
3,01004,-23.549829,-46.634792,Sao Paulo,SP
4,01005,-23.549547,-46.636406,Sao Paulo,SP


In [28]:
geo_clean["geolocation_state"].unique()

<StringArray>
['SP', 'RJ', 'ES', 'MG', 'BA', 'SE', 'PE', 'RN', 'AL', 'PB', 'CE', 'PI', 'MA',
 'PA', 'AP', 'AM', 'RR', 'AC', 'DF', 'GO', 'RO', 'TO', 'MT', 'MS', 'PR', 'SC',
 'RS']
Length: 27, dtype: str

In [29]:
geo_clean["geolocation_city"].unique()

<StringArray>
[             'Sao Paulo',                 'Osasco',            'Carapicuiba',
                'Barueri',    'Santana De Parnaiba',  'Pirapora Do Bom Jesus',
                'Jandira',                'Itapevi',                  'Cotia',
 'Vargem Grande Paulista',
 ...
        'Ipiranga Do Sul',                'Estacao',                 'Ibiaca',
   'Santa Cecilia Do Sul',           'Vila Langaro',                'Charrua',
             'Agua Santa',                'Ciriaco',        'David Canabarro',
              'Muliterno']
Length: 5771, dtype: str

In [30]:
geo_clean = geo_clean[(geo_clean["geolocation_lat"].between(-35, 5)) &
                      (geo_clean["geolocation_lng"].between(-75, 30)) ]

In [31]:
geo_clean.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01001,-23.550227,-46.634039,Sao Paulo,SP
1,01002,-23.547657,-46.634991,Sao Paulo,SP
2,01003,-23.549000,-46.635582,Sao Paulo,SP
3,01004,-23.549829,-46.634792,Sao Paulo,SP
4,01005,-23.549547,-46.636406,Sao Paulo,SP


In [32]:
geo_clean.count()

geolocation_zip_code_prefix    19005
geolocation_lat                19005
geolocation_lng                19005
geolocation_city               19005
geolocation_state              19005
dtype: int64

In [33]:
geo_clean.to_csv("../clean/geolocation.csv", index = False, encoding = "utf-8")

# Order items

In [34]:
order_items = pd.read_csv("../data/olist_order_items_dataset.csv")

In [35]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [36]:
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


In [37]:
# Check null
order_items.isnull().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [38]:
order_items.count()

order_id               112650
order_item_id          112650
product_id             112650
seller_id              112650
shipping_limit_date    112650
price                  112650
freight_value          112650
dtype: int64

In [39]:
# Check duplicate
order_items.duplicated().sum()

np.int64(0)

In [40]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [41]:
#  Chuyển đổi kiểu dữ liệu
order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"])

In [42]:
# Lọc ra data chuẩn
order_items = order_items[(order_items["price"] > 0) & (order_items["freight_value"] > 0)]

In [43]:
order_items.count()

order_id               112267
order_item_id          112267
product_id             112267
seller_id              112267
shipping_limit_date    112267
price                  112267
freight_value          112267
dtype: int64

In [44]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [45]:
order_items.to_csv("../clean/order_items.csv", index = False, encoding = "utf-8")

# Payments

In [46]:
payments = pd.read_csv("../data/olist_order_payments_dataset.csv")

In [47]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [48]:
payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB


In [49]:
# Check null
payments.isnull().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [50]:
# Check duplicate
payments.duplicated().sum()

np.int64(0)

In [51]:
payments.count()

order_id                103886
payment_sequential      103886
payment_type            103886
payment_installments    103886
payment_value           103886
dtype: int64

In [52]:
payments["payment_type"].unique()

<StringArray>
['credit_card', 'boleto', 'voucher', 'debit_card', 'not_defined']
Length: 5, dtype: str

In [53]:
# Xóa khoảng trắng
payments["payment_type"] = payments["payment_type"].str.strip()

In [54]:
payments = payments[payments["payment_value"] >= 0]

In [55]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [56]:
payments.to_csv("../clean/payments.csv", index = False, encoding = "utf-8")

# Reviews

In [57]:
reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")

In [58]:
reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [59]:
reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


In [60]:
# Check Duplicate
reviews.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [61]:
reviews.drop(columns = ["review_comment_title"], inplace = True)

In [62]:
reviews["review_comment_message"][reviews["review_comment_message"].notnull()]

3                    Recebi bem antes do prazo estipulado.
4        Parabéns lojas lannister adorei comprar pela I...
9        aparelho eficiente. no site a marca do aparelh...
12         Mas um pouco ,travando...pelo valor ta Boa.\r\n
15       Vendedor confiável, produto ok e entrega antes...
16       GOSTARIA DE SABER O QUE HOUVE, SEMPRE RECEBI E...
19                                                 Péssimo
22                                            Loja nota 10
24                   obrigado pela atençao amim dispensada
27       A compra foi realizada facilmente.\r\nA entreg...
28                          relógio muito bonito e barato.
29                     Não gostei ! Comprei gato por lebre
32       Sempre compro pela Internet e a entrega ocorre...
34       Recebi exatamente o que esperava. As demais en...
36                                             Recomendo ,
37                                              muito boa 
38       Tô completamente apaixonada, loja super respon.

In [63]:
# Chuyển đổi kiểu dữ liệu cột review_creation_date, review_answer_timestamp        
reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"])
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])

In [64]:
reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                99224 non-null  str           
 1   order_id                 99224 non-null  str           
 2   review_score             99224 non-null  int64         
 3   review_comment_message   40977 non-null  str           
 4   review_creation_date     99224 non-null  datetime64[us]
 5   review_answer_timestamp  99224 non-null  datetime64[us]
dtypes: datetime64[us](2), int64(1), str(3)
memory usage: 4.5 MB


In [65]:
# Check duplicate
reviews.drop_duplicates().count()

review_id                  99224
order_id                   99224
review_score               99224
review_comment_message     40977
review_creation_date       99224
review_answer_timestamp    99224
dtype: int64

In [66]:
# Lọc các data bẩn trong cột review_comment_message
reviews["review_comment_message"] = reviews["review_comment_message"].str.replace(r"\r|\n|\?", "", regex = True)

In [71]:
# Fill giá trị null 
reviews.fillna(value = {"review_comment_message" : "No Comment"}, inplace = True).head()

,review_id,order_id,review_score,review_comment_message,review_creation_date,review_answer_timestamp,review_sentiment
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,No Comment,2018-01-18,2018-01-18 21:46:59,Positive
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,No Comment,2018-03-10,2018-03-11 03:05:13,Positive
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,No Comment,2018-02-17,2018-02-18 14:36:24,Positive
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06,Positive
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53,Positive


In [68]:
# Thêm cột nhãn sentiment
reviews["review_sentiment"] = reviews["review_score"].apply(lambda x : "Positive" if x >= 4 
                                                            else "Neutral" if x >= 3 
                                                            else "Negative")

In [72]:
reviews.head()

,review_id,order_id,review_score,review_comment_message,review_creation_date,review_answer_timestamp,review_sentiment
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,No Comment,2018-01-18,2018-01-18 21:46:59,Positive
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,No Comment,2018-03-10,2018-03-11 03:05:13,Positive
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,No Comment,2018-02-17,2018-02-18 14:36:24,Positive
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06,Positive
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53,Positive


In [70]:
reviews["review_score"].unique()

array([4, 5, 1, 3, 2])

In [73]:
reviews.to_csv("../clean/reviews.csv", index = False, encoding = "utf-8")

# Orders

In [106]:
orders = pd.read_csv("../data/olist_orders_dataset.csv")

In [75]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [78]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [107]:
# Check null
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [81]:
# Check duplicate
orders.duplicated().sum()

np.int64(0)

In [108]:
# Chuyển đổi kiểu dữ liệu cho các cột date
date_col = ["order_purchase_timestamp",	"order_approved_at",	"order_delivered_carrier_date",	"order_delivered_customer_date",	"order_estimated_delivery_date"]
for col in date_col :
    orders[col] = pd.to_datetime(orders[col])

In [83]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


In [ ]:
# Validate data
orders = orders[
    (
        (orders["order_approved_at"] >= orders["order_purchase_timestamp"]) &
        (orders["order_delivered_customer_date"] >= orders["order_approved_at"])
    )
    |
    (orders["order_approved_at"].isnull())
    |
    (orders["order_purchase_timestamp"].isnull())
    |
    (orders["order_delivered_customer_date"].isnull())
]

In [111]:
orders.count()

order_id                         99380
customer_id                      99380
order_status                     99380
order_purchase_timestamp         99380
order_approved_at                99220
order_delivered_carrier_date     97597
order_delivered_customer_date    96415
order_estimated_delivery_date    99380
dtype: int64

In [114]:
orders.to_csv("../clean/orders.csv", index = False, encoding = "utf-8")

# Products

In [115]:
products = pd.read_csv("../data/olist_products_dataset.csv")

In [116]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [118]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB


In [ ]:
# Check null 
products.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [124]:
# Check duplicate
products.duplicated().sum()

np.int64(0)

In [ ]:
# Loại bỏ khoảng trắng
products["product_category_name"] = products["product_category_name"].str.strip()

In [132]:
products.sort_values(
    "product_weight_g",
    ascending=False
).head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
25166,26644690fde745fc4654719c3904e1db,cama_mesa_banho,59.0,534.0,1.0,40425.0,13.0,65.0,28.0
11327,8d4e92265a16e69a1e1d76e67e46d72f,instrumentos_musicais,45.0,422.0,5.0,30000.0,80.0,75.0,36.0
31042,5459103ed3b2f852d267e6a70b00dc24,casa_construcao,56.0,706.0,1.0,30000.0,40.0,79.0,40.0
5783,b2b7d701c1f12d5a0f52ec90b0a8f819,beleza_saude,58.0,424.0,1.0,30000.0,55.0,75.0,61.0
27230,e5f2d52b802189ee658865ca93d83a8f,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0
